# Case study STGAN: 12 giugno e 2–3 luglio 2019 a t+1/t+6

Analisi esclusivamente **post-hoc**. Il notebook usa le decisioni `is_anomaly` già esportate da STGAN e le previsioni SDE-Net già associate all'esatto `(location, timestamp target)`. Non riaddestra STGAN o SDE-Net.

I due eventi sono scelti con criteri dichiarati: il 12 giugno ha il maggior numero giornaliero di flag STGAN; il 2 luglio ha la massima copertura spaziale istantanea e l'episodio continua il 3 luglio.

In [ ]:
import importlib, json, os, sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display

ROOT = Path.cwd().resolve()
if not (ROOT / 'physiq_pv').is_dir():
    for parent in ROOT.parents:
        if (parent / 'physiq_pv').is_dir():
            ROOT = parent
            break
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from physiq_pv.experiments import sde_pipeline as pipe
from physiq_pv.reporting import anomaly_driver, anomaly_extremes
anomaly_driver = importlib.reload(anomaly_driver)
anomaly_extremes = importlib.reload(anomaly_extremes)

STGAN_SEED = 20
HORIZONS = (1, 6)
STGAN_SEED_DIR = Path(os.environ.get(
    'STGAN_SEED_DIR',
    ROOT / 'outputs' / 'pvgis_stgan' / 'paper_reference' / f'seed_{STGAN_SEED}',
)).resolve()
STGAN_SCORES = STGAN_SEED_DIR / 'anomaly_scores.csv'
EVALUATION_DIR = Path(os.environ.get(
    'STGAN_POSTHOC_ROOT',
    ROOT / 'outputs' / f'sde_stgan_direct_multihorizon_seed{STGAN_SEED}',
)).resolve()
PREDICTIONS = EVALUATION_DIR / 'predictions.csv'
OUT_DIR = Path(os.environ.get(
    'STGAN_RARE_EVENTS_OUT_DIR',
    EVALUATION_DIR / 'stgan_rare_events_t1_t6',
)).resolve()
FIGURE_DIR = OUT_DIR / 'figures'
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
REFERENCE_PEAK_BY_HORIZON = {
    horizon: EVALUATION_DIR / 'posthoc_by_horizon' / f't_plus_{horizon}' / 'reference_production_peaks.csv'
    for horizon in HORIZONS
}
EVENTS = {
    'june_12_stgan_peak': {
        'days': ('2019-06-12',),
        'context_start': '2019-06-11',
        'context_end_exclusive': '2019-06-14',
        'selection': 'massimo numero giornaliero di flag STGAN',
    },
    'july_02_03_stgan_regional': {
        'days': ('2019-07-02', '2019-07-03'),
        'context_start': '2019-07-01',
        'context_end_exclusive': '2019-07-05',
        'selection': 'massima copertura spaziale il 2 luglio, persistente il 3 luglio',
    },
}
CSV_CHUNKSIZE = 500_000
print('STGAN scores :', STGAN_SCORES)
print('Predizioni   :', PREDICTIONS)
print('Output       :', OUT_DIR)

## 1. Controllo delle sorgenti

Prima eseguire `stgan_pointwise_posthoc_sdenet.ipynb`, che crea le predizioni evaluation-only e i riferimenti di produzione separati per t+1 e t+6.

In [ ]:
required_paths = [
    STGAN_SCORES, PREDICTIONS, EVALUATION_DIR / 'evaluation_source.json',
    *REFERENCE_PEAK_BY_HORIZON.values(),
]
missing = [path for path in required_paths if not path.is_file()]
if missing:
    raise FileNotFoundError(
        'File mancanti; eseguire prima stgan_pointwise_posthoc_sdenet.ipynb:\n'
        + '\n'.join(map(str, missing))
    )
metadata = json.loads(
    (EVALUATION_DIR / 'evaluation_source.json').read_text(encoding='utf-8')
)
if metadata.get('detector') != 'stgan':
    raise ValueError(f"Evaluation detector non STGAN: {metadata.get('detector')!r}")
score_header = set(pd.read_csv(STGAN_SCORES, nrows=0).columns)
if not {'location', 'timestamp', 'anomaly_score', 'is_anomaly'} <= score_header:
    raise ValueError('anomaly_scores.csv STGAN non ha lo schema atteso.')
prediction_header = set(pd.read_csv(PREDICTIONS, nrows=0).columns)
required_prediction_columns = {
    'location', 'timestamp', 'horizon_hours', 'anomaly_group', 'y_true',
    'lower_pi', 'upper_pi', 'solar_irradiance_poa_target',
}
if not required_prediction_columns <= prediction_header:
    raise ValueError(
        f'Colonne predizioni mancanti: {sorted(required_prediction_columns - prediction_header)}'
    )
if not ({'y_pred', 'y_pred_mean'} & prediction_header):
    raise ValueError('Le predizioni richiedono y_pred oppure y_pred_mean.')
if 'event_group' in prediction_header:
    raise ValueError('Le predizioni contengono event_group: non sono STGAN-only.')
available_horizons = tuple(sorted(
    int(value) for value in pd.read_csv(
        PREDICTIONS, usecols=['horizon_hours']
    )['horizon_hours'].drop_duplicates()
))
if not set(HORIZONS) <= set(available_horizons):
    raise ValueError(f'Orizzonti richiesti {HORIZONS}, disponibili {available_horizons}.')
display(pd.DataFrame([metadata]))
print('OK: label STGAN puntuali e forecast direct t+1/t+6 disponibili.')

## 2. Verifica data-driven degli eventi

La classifica usa direttamente `is_anomaly`. Il numero di flag misura la persistenza complessiva; la quota massima oraria misura l'estensione spaziale simultanea.

In [ ]:
regional = anomaly_extremes.regional_flag_series(
    STGAN_SCORES, chunksize=2_000_000
)
daily = anomaly_extremes.rank_flagged_days(regional)
daily.to_csv(OUT_DIR / 'stgan_daily_anomaly_ranking.csv', index=False)
top_days = daily.sort_values(
    ['n_anomalies', 'anomaly_share'], ascending=False
).head(15)
selected_dates = tuple(
    day for event in EVENTS.values() for day in event['days']
)
selected_daily = daily[daily['day'].isin(pd.to_datetime(selected_dates))].copy()
display(Markdown('### Quindici giorni STGAN più anomali'))
display(top_days[[
    'day', 'n_anomalies', 'anomaly_share', 'n_active_hours', 'score_max',
]])
display(Markdown('### Giorni scelti per i case study'))
display(selected_daily[[
    'day', 'n_anomalies', 'anomaly_share', 'n_active_hours', 'score_max',
]])

fig, axes = plt.subplots(len(EVENTS), 1, figsize=(14, 7), squeeze=False)
for axis, (event_name, event) in zip(axes.flat, EVENTS.items()):
    start = pd.Timestamp(event['context_start'])
    end = pd.Timestamp(event['context_end_exclusive'])
    frame = regional.loc[(regional.index >= start) & (regional.index < end)]
    axis.plot(frame.index, frame['extreme_share'], color='tab:red', marker='o')
    for day in event['days']:
        axis.axvspan(pd.Timestamp(day), pd.Timestamp(day) + pd.Timedelta(days=1), color='red', alpha=0.10)
    axis.set(title=event_name, ylabel='Località anomale')
    axis.grid(alpha=0.25)
axes[-1, 0].set_xlabel('Timestamp STGAN / target SDE-Net')
fig.suptitle('Estensione regionale delle anomalie STGAN')
fig.tight_layout()
regional_figure = FIGURE_DIR / 'stgan_selected_events_regional_share.png'
fig.savefig(regional_figure, dpi=180, bbox_inches='tight')
plt.show()

## 3. Label puntuali nei soli giorni evento

Le righe sono selezionate in streaming dal CSV STGAN. Una previsione entra nella categoria rara soltanto quando la stessa località e lo stesso timestamp target hanno `is_anomaly=True`.

In [ ]:
def _boolean_flags(values):
    flags = values.astype(str).str.strip().str.lower().map(
        {'true': True, 'false': False, '1': True, '0': False}
    )
    if flags.isna().any():
        raise ValueError('is_anomaly STGAN contiene valori non booleani.')
    return flags.astype(bool)

event_by_day = {
    pd.Timestamp(day).normalize(): event_name
    for event_name, event in EVENTS.items() for day in event['days']
}
label_parts = []
for chunk in pd.read_csv(
    STGAN_SCORES, usecols=['location', 'timestamp', 'is_anomaly'],
    dtype={'location': 'string'}, chunksize=CSV_CHUNKSIZE,
):
    timestamp = pd.to_datetime(chunk['timestamp'], errors='coerce', utc=True).dt.tz_convert(None)
    day = timestamp.dt.normalize()
    selected = day.isin(event_by_day) & _boolean_flags(chunk['is_anomaly'])
    if not selected.any():
        continue
    part = pd.DataFrame({
        'location': chunk.loc[selected, 'location'].astype(str).to_numpy(),
        'timestamp': timestamp.loc[selected].to_numpy(),
        'category': day.loc[selected].map(event_by_day).to_numpy(),
    })
    label_parts.append(part)
if not label_parts:
    raise ValueError('Nessun flag STGAN trovato nei giorni selezionati.')
event_labels = pd.concat(label_parts, ignore_index=True)
if event_labels.duplicated(['location', 'timestamp']).any():
    raise ValueError('Label STGAN duplicate per location-timestamp.')
event_labels.to_csv(OUT_DIR / 'stgan_event_pointwise_labels.csv', index=False)
display(event_labels.groupby('category').agg(
    n_flags=('location', 'size'), n_locations=('location', 'nunique'),
))

## 4. Risposta temporale SDE-Net a t+1 e t+6

Ogni pannello usa tutte le località, mostra previsione, target, intervallo predittivo, MAE/RMSE, PICP e incertezza. Le date evidenziate sono timestamp target, non timestamp di emissione.

In [ ]:
diagnostics = {}
diagnostic_summaries = []
for event_name, event in EVENTS.items():
    for horizon_hours in HORIZONS:
        result = pipe.build_extreme_event_diagnostic(
            str(EVALUATION_DIR),
            start=event['context_start'],
            end=event['context_end_exclusive'],
            figure_subdir=f'events/stgan_rare_t1_t6/{event_name}/t_plus_{horizon_hours}',
            horizon_hours=horizon_hours,
        )
        diagnostics[(event_name, horizon_hours)] = result
        summary = result['summary'].copy()
        summary.insert(0, 'event', event_name)
        diagnostic_summaries.append(summary)
        display(Markdown(f'### {event_name} — forecast t+{horizon_hours}'))
        display(summary)
        display(Image(filename=str(result['figure_path'])))
diagnostic_summary = pd.concat(diagnostic_summaries, ignore_index=True)
diagnostic_summary.to_csv(OUT_DIR / 'temporal_diagnostic_summary.csv', index=False)

## 5. Confronto puntuale: normale contro flag STGAN

Questo confronto risponde alla domanda: quanto sbaglia SDE-Net esattamente nelle località e nei timestamp che STGAN considera rari, rispetto alle righe normali degli stessi giorni?

In [ ]:
pointwise_results = {}
pointwise_metrics = []
for horizon_hours in HORIZONS:
    result = anomaly_driver.build_anomaly_driver_comparison_figures(
        EVALUATION_DIR, event_labels,
        figure_subdir=f'events/stgan_rare_t1_t6/pointwise/t_plus_{horizon_hours}',
        metrics_name=f'stgan_rare_events_pointwise_t_plus_{horizon_hours}.csv',
        restrict_days=selected_dates,
        figures_per_category=True,
        horizon_hours=horizon_hours,
        reference_peak_path=REFERENCE_PEAK_BY_HORIZON[horizon_hours],
    )
    pointwise_results[horizon_hours] = result
    metrics = result['metrics'].copy()
    metrics['horizon_hours'] = horizon_hours
    pointwise_metrics.append(metrics)
    display(Markdown(f'### Metriche puntuali STGAN — t+{horizon_hours}'))
    display(metrics)
    for figure_path in result['figure_paths'].values():
        display(Image(filename=str(figure_path)))
pointwise_metrics = pd.concat(pointwise_metrics, ignore_index=True)
pointwise_metrics.to_csv(OUT_DIR / 'pointwise_metrics_t1_t6.csv', index=False)

## 6. Confronto dell'intero evento con il normale 2019

Qui entrano tutte le località dei giorni evento, anche quelle non flaggate da STGAN. Il baseline contiene le righe che STGAN considera normali nel resto del 2019. Le metriche sono separate per fascia di produzione e orizzonte.

In [ ]:
calendar_results = {}
calendar_metrics = []
for event_name, event in EVENTS.items():
    for horizon_hours in HORIZONS:
        result = pipe.build_extreme_event_comparison_figures(
            str(EVALUATION_DIR),
            event_dates=tuple(event['days']),
            comparison_name=f'stgan_{event_name}_t_plus_{horizon_hours}',
            figure_subdir=f'events/stgan_rare_t1_t6/{event_name}/t_plus_{horizon_hours}/calendar',
            horizon_hours=horizon_hours,
            reference_peak_path=REFERENCE_PEAK_BY_HORIZON[horizon_hours],
        )
        calendar_results[(event_name, horizon_hours)] = result
        metrics = result['metrics'].copy()
        metrics['event'] = event_name
        metrics['horizon_hours'] = horizon_hours
        calendar_metrics.append(metrics)
        display(Markdown(f'### {event_name} — calendario t+{horizon_hours}'))
        for metric_name in ('mae', 'rmse', 'picp', 'nmpil', 'clc'):
            display(Markdown(f'**{metric_name.upper()} per bin**'))
            display(metrics.pivot(index='bin', columns='category', values=metric_name))
        for figure_path in result['figure_paths'].values():
            display(Image(filename=str(figure_path)))
calendar_metrics = pd.concat(calendar_metrics, ignore_index=True)
calendar_metrics.to_csv(OUT_DIR / 'calendar_event_metrics_t1_t6.csv', index=False)

## 7. Confronto diretto t+1 contro t+6

In [ ]:
for event_name in EVENTS:
    display(Markdown(f'### {event_name}'))
    event_metrics = calendar_metrics[calendar_metrics['event'].eq(event_name)]
    for metric_name in ('mae', 'rmse'):
        display(Markdown(f'**{metric_name.upper()}: t+1 vs t+6**'))
        display(event_metrics.pivot_table(
            index='bin', columns=['horizon_hours', 'category'],
            values=metric_name, aggfunc='first',
        ))

analysis_metadata = {
    'analysis': 'stgan_rare_events_t1_t6',
    'post_processing_only': True,
    'training_rerun': False,
    'detector': 'stgan',
    'label_source': 'saved_is_anomaly',
    'join_key': ['location', 'timestamp'],
    'timestamps_are_target_times': True,
    'horizons_hours': list(HORIZONS),
    'events': EVENTS,
    'stgan_scores': str(STGAN_SCORES),
    'predictions': str(PREDICTIONS),
}
(OUT_DIR / 'analysis_metadata.json').write_text(
    json.dumps(analysis_metadata, indent=2), encoding='utf-8'
)
print('Output principali:')
for path in sorted(OUT_DIR.iterdir()):
    print(' -', path.name)

## Interpretazione

- Il confronto **puntuale** misura l'errore proprio dove STGAN assegna `is_anomaly=True`.
- Il confronto **calendar event** misura l'impatto dell'intero giorno o episodio, comprese le località non flaggate.
- Il confronto `t+1`/`t+6` mostra se l'evento diventa più difficile aumentando l'orizzonte.
- Il 12 giugno rappresenta il massimo giornaliero per numero di flag; il 2–3 luglio rappresenta l'episodio con la maggiore estensione spaziale e persistenza.

STGAN resta un detector non supervisionato: `raro` non significa automaticamente evento meteorologico reale né forecast difficile.